# Meridian Backtester — Results

Comparison of three signal sources over the 2010–2024 window using the
Jan 2010 S&P 100 (OEX) universe, daily bars, 5 bps slippage each side.

Regime splits: **Pre-COVID** 2010-01-01 → 2020-02-29 · **Post-COVID** 2020-03-01 → 2024-12-31

*Generated by `backtest/run_quant.py`. Data from yfinance with `auto_adjust=True`.*

In [ ]:
from __future__ import annotations
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolve repo root so this notebook works from any working directory.
_here = Path().resolve()
_root = _here
while not (_root / "backtest").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

from backtest.notebooks.nb_utils import (
    METRIC_KEYS, METRIC_LABELS, METRIC_FORMATS, WINDOW_LABELS,
    build_metrics_comparison,
    compute_drawdown_series,
    format_metric_val,
    load_spy_results,
    load_strategy_results,
    normalise_equity,
    regime_sharpe_df,
)

_RESULTS_BASE = _root / "backtest" / "results"
_STRATS       = ["quant", "vader", "finbert"]
_COLOURS      = {"quant": "#1f77b4", "vader": "#ff7f0e", "finbert": "#2ca02c", "spy": "#888888"}
_LABELS       = {"quant": "Quant", "vader": "VADER", "finbert": "FinBERT", "spy": "SPY B&H"}

# ── Load ──────────────────────────────────────────────────────────────────────
all_results = {s: load_strategy_results(s, _RESULTS_BASE) for s in _STRATS}
spy_df      = load_spy_results(_RESULTS_BASE)

available = [s for s in _STRATS if all_results[s] is not None]
print(f"Loaded: {available or 'none yet'}")

## Data Availability

In [ ]:
_STATUS = {True: "✓ results available", False: "✗ not yet (see TODO.md)"}
rows = []
for s in _STRATS:
    rows.append({"Signal": _LABELS[s], "Status": _STATUS[all_results[s] is not None]})
rows.append({"Signal": "SPY benchmark", "Status": _STATUS[spy_df is not None]})
display(pd.DataFrame(rows).set_index("Signal"))

## Full-Window Metrics (2010–2024)

'—' indicates the strategy has not been run yet or produced no trades.
Sharpe SE uses the Mertens/Lo (2002) iid approximation; true SE is wider
for strategies with autocorrelated returns.

In [ ]:
def _fmt_table(df: pd.DataFrame) -> pd.DataFrame:
    """Format each cell using the metric-specific format string."""
    out = {}
    for metric, label in zip(METRIC_KEYS, METRIC_LABELS.values()):
        row = {}
        for col in df.columns:
            row[col] = format_metric_val(df.loc[label, col], metric)
        out[label] = row
    return pd.DataFrame(out).T

comparison = build_metrics_comparison(all_results, window="full_window")
display(_fmt_table(comparison))

## Equity Curves (Rebased to 100)

All strategies start at 100 regardless of initial cash so absolute returns
are directly comparable.  The red dotted line marks the COVID regime split
(2020-03-01).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for s in available:
    normed = normalise_equity(all_results[s]["equity_curve"])
    ax.plot(normed.index, normed.values,
            label=_LABELS[s], color=_COLOURS[s], linewidth=1.5)

if spy_df is not None:
    normed_spy = normalise_equity(spy_df)
    ax.plot(normed_spy.index, normed_spy.values,
            label=_LABELS["spy"], color=_COLOURS["spy"],
            linewidth=1.2, linestyle="--", alpha=0.8)

ax.axvline(pd.Timestamp("2020-03-01"), color="red",
           linestyle=":", linewidth=1.2, label="COVID split")

if not available and spy_df is None:
    ax.text(0.5, 0.5, "No results available yet\nRun backtest first",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=14, color="gray")

ax.set_title("Equity Curves (rebased to 100)", fontsize=13)
ax.set_ylabel("Portfolio value (rebased)")
ax.set_xlabel("")
ax.legend(loc="upper left")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
plt.tight_layout()
plt.show()

## Drawdown

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for s in available:
    dd = compute_drawdown_series(all_results[s]["equity_curve"]) * 100
    ax.fill_between(dd.index, dd.values, 0,
                    alpha=0.35, color=_COLOURS[s], label=_LABELS[s])
    ax.plot(dd.index, dd.values, color=_COLOURS[s], linewidth=0.8)

ax.axvline(pd.Timestamp("2020-03-01"), color="red",
           linestyle=":", linewidth=1.2, label="COVID split")

if not available:
    ax.text(0.5, 0.5, "No results available yet",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=14, color="gray")

ax.set_title("Drawdown (%)", fontsize=13)
ax.set_ylabel("Drawdown (%)")
ax.set_xlabel("")
ax.legend(loc="lower left")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
plt.tight_layout()
plt.show()

## Regime Split: Sharpe Ratio

Pre-COVID (2010–Feb 2020) vs Post-COVID (Mar 2020–2024).
A strategy with consistent Sharpe across both halves is more robust than
one that only looks good in one regime.

In [ ]:
rsdf = regime_sharpe_df(all_results)
display(rsdf)

# Bar chart — only plot if at least one strategy has results
if available:
    x      = range(len(_STRATS))
    width  = 0.25
    labels = [_LABELS[s] for s in _STRATS]

    fig, ax = plt.subplots(figsize=(9, 4))
    for i, (wk, lbl) in enumerate(WINDOW_LABELS.items()):
        vals = []
        for s in _STRATS:
            v = rsdf.loc[s.capitalize(), lbl] if s.capitalize() in rsdf.index else None
            vals.append(float(v) if v is not None else 0.0)
        ax.bar([xi + i * width for xi in x], vals,
               width=width, label=lbl, alpha=0.8)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks([xi + width for xi in x])
    ax.set_xticklabels(labels)
    ax.set_title("Sharpe Ratio by Regime", fontsize=13)
    ax.set_ylabel("Sharpe ratio")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Trade Statistics

Hit rate, average holding period, and trade count per strategy.
Quant (momentum + mean-reversion) typically shows shorter holding periods
than sentiment-based strategies.

In [ ]:
if available:
    trade_rows = []
    for s in available:
        trades = all_results[s]["trades"]
        m      = all_results[s]["metrics"]["full_window"]
        trade_rows.append({
            "Signal":       _LABELS[s],
            "# Trades":     format_metric_val(m.get("trade_count"), "trade_count"),
            "Hit Rate":     format_metric_val(m.get("hit_rate"),    "hit_rate"),
            "Avg Win ($)":  format_metric_val(m.get("avg_win"),     "avg_win"),
            "Avg Loss ($)": format_metric_val(m.get("avg_loss"),    "avg_loss"),
            "W/L Ratio":    format_metric_val(m.get("win_loss_ratio"), "win_loss_ratio"),
            "Avg Hold (d)": format_metric_val(m.get("avg_holding_days"), "avg_holding_days"),
        })
    display(pd.DataFrame(trade_rows).set_index("Signal"))
else:
    print("No results yet — run the backtests first.")

## Pre-COVID vs Post-COVID — Full Metrics

In [ ]:
for window, label in WINDOW_LABELS.items():
    print(f"\n### {label}")
    cmp = build_metrics_comparison(all_results, window=window)
    display(_fmt_table(cmp))

## Honest Disclosures

1. **Daily bars from yfinance** — `auto_adjust=True` for total-return semantics.
   Known caveats: adjusted-close handling may differ for stocks with complex
   corporate actions; occasional gaps on illiquid/delisted names; dividend
   treatment varies. Spot-check accordingly.

2. **5 bps slippage assumption** is stylised; actual slippage varies by ticker
   liquidity. The 2010–2024 era is post-Reg-NMS and post-decimalization,
   making a single slippage assumption more defensible than spanning 2005–2007.

3. **Universe is S&P 100 (OEX) constituents as of Jan 2010, held constant.**
   A handful of names were acquired or delisted during 2010–2024.
   Post-event returns are recorded honestly: acquisitions at cash price,
   bankruptcies at −100%, no resurrection. This is the survivorship-bias-free
   universe.

4. **No commission model.** Modern retail brokers are commission-free.
   Pre-2019 retail commissions ($5–$10 per trade) are NOT modelled — disclose
   this when reporting pre-2019 returns. Institutional commissions (1–3 bps)
   are folded into the 5 bps slippage assumption.

5. **Sentiment data source: FNSPID (Dong et al. 2024).** VADER and FinBERT
   use the FNSPID academic news dataset
   ([arXiv:2402.06698](https://arxiv.org/abs/2402.06698),
   HuggingFace `Zihan1004/FNSPID`): 15.7M timestamp-aligned headlines for
   4,775 S&P 500 companies, 1999–2023. **License: CC BY-NC 4.0** — this
   backtest is research/non-commercial use; a commercial product would
   require separate licensing.

6. **Window asymmetry: sentiment ends 2023-12-31.** FNSPID coverage stops
   at end of 2023 while yfinance bars extend through 2024-12-31. For
   apples-to-apples comparison in the full-window cell above, the quant
   curve should be sliced to 2023-12-31 when comparing against sentiment
   strategies. The pre-COVID window (2010–Feb 2020) is unaffected.

7. **FinBERT thresholds match the live bot** (`finbert_threshold=0.6`,
   `vader_threshold=0.35`,  `min_count=2`, `min_consensus=0.6`) so the
   backtest measures the same decision boundary that ran in production.